In [83]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from core.signal.preprocess import *
from glob import glob
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
collectedCSV  = glob("vctk/gen/csv/wav48_silence_trimmed/*/*.csv")
collectedFLAC = [csv.replace("vctk/gen/csv", "vctk/VCTK-Corpus-0.92").replace("csv", "flac") for csv in collectedCSV]
Fs = 16000

In [ ]:
vctkds = torchaudio.datasets.VCTK_092(root="vctk/", download=False)
maxLen = max([e[0].shape[1] for e in vctkds])

In [26]:
for fn in tqdm(collectedFLAC):
    x, sr = torchaudio.load(fn)
    tform = torchaudio.transforms.Resample(sr, Fs)
    x = tform(x)
    # pad to 226209
    x = F.pad(x, (0, 226209 - x.shape[1]))
    newFn = fn.replace("vctk/VCTK-Corpus-0.92", "vctk/resampled/VCTK-Corpus-0.92")
    # make sure the directory exists
    os.makedirs(os.path.dirname(newFn), exist_ok=True)
    torchaudio.save(newFn, x, Fs)

  0%|          | 0/8942 [00:00<?, ?it/s]

100%|██████████| 8942/8942 [01:06<00:00, 133.65it/s]


In [57]:
vctkds = torchaudio.datasets.VCTK_092(root="vctk/resampled/", download=False)
# Split into train and test
split = 0.85
lenTrain = int(len(vctkds) * split)
lenTest = len(vctkds) - lenTrain
train, test = torch.utils.data.random_split(vctkds, [len(vctkds) - lenTest, lenTest])
train_loader = torch.utils.data.DataLoader(train, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=16, shuffle=False)

In [86]:
speaker_list = ['p225', 'p226','p227','p228', 'p229', 'p230', 'p231', 'p232', 'p233', 'p234', 'p236', 'p237', 'p238', 'p239', 'p240', 'p241', 'p243', 'p244', 'p245', 'p246', 'p247', 'p248', 'p249']
nSpeakers = len(speaker_list)
onehot_speaker = lambda x: torch.eye(nSpeakers)[speaker_list.index(x)]
# Is this a clean way to do this? Hell nah
# Is this efficient? Yes

In [82]:
for (waveform, _, _, speaker_id, _) in train_loader:
    speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
    print(waveform.shape)
    break

torch.Size([16, 1, 226209])


In [84]:
class ConvNet2D(nn.Module):
    def __init__(self, nSpeakers):
        super(ConvNet2D, self).__init__()
        self.spec = torchaudio.transforms.Spectrogram(n_fft=1024, hop_length=256, win_length=1024, power=1)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(9, 9), stride=(1, 1))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(7, 7), stride=(1, 1))
        self.conv3 = nn.Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1))
        self.conv4 = nn.Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
        self.conv5 = nn.Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
    
    def forward(self, x):
        x = self.spec(x)
        return x

In [87]:
net = ConvNet2D(nSpeakers)
net(waveform).shape

torch.Size([16, 1, 513, 884])